# Módulo 2 — GitHub, CTEs, Window Functions y EDA
**Bootcamp: Fundamentos de Ingeniería de Datos**

En este módulo trabajaremos con un dataset de propiedades inmobiliarias argentinas cargado en una tabla Delta gestionada por Unity Catalog.

---

## Contenido del módulo
1. **EDA** — Exploración y análisis descriptivo del dataset
2. **CTEs** — Common Table Expressions (WITH)
3. **Window Functions** — Funciones de ventana (OVER, PARTITION BY)

---

**Tabla:** `bootcamp.bronze.properties_bronze` (Unity Catalog)
- 1,246,717 filas × 22 columnas
- Fuente: Listings inmobiliarios (zonaprop.com.ar)

## ⚙ SETUP — Verificar tabla bronze en Unity Catalog

El dataset fue cargado en la tabla `bootcamp.bronze.properties_bronze` desde el volume `bootcamp.practice.modulo_2_data`.
Verificamos que la tabla existe y tiene datos antes de comenzar los ejercicios.

In [0]:
# Preparación: Crear schemas y tabla bronze
spark.sql("CREATE SCHEMA IF NOT EXISTS bootcamp.landing")
print("✅ Schema bootcamp.landing creado")

spark.sql("CREATE SCHEMA IF NOT EXISTS bootcamp.bronze")
print("✅ Schema bootcamp.bronze creado")

# Crear tabla bronze desde el CSV en el volume (si no existe)
df = spark.read.csv(
    "/Volumes/bootcamp/practice/modulo_2_data/properties_raw.csv",
    header=True,
    inferSchema=True
)
df.write.saveAsTable("bootcamp.bronze.properties_bronze", mode="overwrite")

# Verificar que la tabla existe y tiene datos
count = spark.sql("SELECT COUNT(*) AS total FROM bootcamp.bronze.properties_bronze").collect()[0]['total']
print(f"✅ Tabla bootcamp.bronze.properties_bronze verificada — {count:,} filas")

## PARTE 1: EDA (Exploratory Data Analysis)

Explorar el dataset para entender su estructura, calidad de datos y distribuciones.

**Instrucciones:** Resolvé cada ejercicio en la celda correspondiente.

In [0]:
%sql
-- Ejercicio 1.1: ¿Cuántas filas y columnas tiene el dataset?
-- Usá DESCRIBE para ver los tipos de datos de cada columna.
-- ¿Qué columnas están como STRING que probablemente deberían ser numéricas?

DESCRIBE bootcamp.bronze.properties_bronze;

-- 1) El dataset tiene 22 columnas y 1.246.717 filas.
-- 2) Las columnas que probablemente deberían ser numéricas son: precio, numero, expensas, ambientes, metros_cuadrados_totales, metros_cuadrados_cubiertos, piso, antiguedad.

In [0]:
%sql
SELECT COUNT(*)
FROM bootcamp.bronze.properties_bronze;

In [0]:
%sql
-- Ejercicio 1.2: Contá cuántos valores NULL tiene cada columna.
-- Ordená de mayor a menor cantidad de nulos.
-- Reflexión: ¿Qué columnas tienen más datos faltantes? ¿Por qué podría ser?

SELECT
    SUM(CASE WHEN id IS NULL THEN 1 ELSE 0 END) AS nulls_id,
    SUM(CASE WHEN precio IS NULL THEN 1 ELSE 0 END) AS nulls_precio,
    SUM(CASE WHEN ambientes IS NULL THEN 1 ELSE 0 END) AS nulls_ambientes,
    SUM(CASE WHEN metros_cuadrados_totales IS NULL THEN 1 ELSE 0 END) AS nulls_metros_totales,
    SUM(CASE WHEN metros_cuadrados_cubiertos IS NULL THEN 1 ELSE 0 END) AS nulls_metros_cubiertos,
    SUM(CASE WHEN expensas IS NULL THEN 1 ELSE 0 END) AS nulls_expensas,
    SUM(CASE WHEN zona IS NULL THEN 1 ELSE 0 END) AS nulls_zona,
    SUM(CASE WHEN tipo_de_operacion IS NULL THEN 1 ELSE 0 END) AS nulls_tipo_operacion
FROM bootcamp.bronze.properties_bronze;

In [0]:
%sql
-- Ejercicio 1.3: ¿Cuántas propiedades hay por zona?
-- Mostrá las 15 zonas con más propiedades.

SELECT zona, COUNT(*) AS total_propiedades
FROM bootcamp.bronze.properties_bronze
GROUP BY zona
ORDER BY total_propiedades DESC
LIMIT 15;

In [0]:
%sql
-- Ejercicio 1.4: ¿Cuántas propiedades hay por tipo de operación (venta/alquiler) y moneda (USD/ARS)?

SELECT
    tipo_de_operacion,
    moneda,
    COUNT(*) AS total
FROM bootcamp.bronze.properties_bronze
GROUP BY tipo_de_operacion, moneda
ORDER BY total DESC;

In [0]:
%sql
-- Ejercicio 1.5: Calculá el precio mínimo, máximo y promedio por moneda.
-- Nota: la columna precio está como STRING — usá CAST(precio AS DOUBLE)
-- Reflexión: ¿Los valores extremos tienen sentido o hay outliers?

SELECT
    moneda,
    MIN(CAST(precio AS DOUBLE)) AS precio_min,
    MAX(CAST(precio AS DOUBLE)) AS precio_max,
    AVG(CAST(precio AS DOUBLE)) AS precio_promedio,
    COUNT(*) AS total
FROM bootcamp.bronze.properties_bronze
WHERE precio IS NOT NULL
GROUP BY moneda
ORDER BY total DESC;

## PARTE 2: CTEs (Common Table Expressions)

Una CTE es una consulta temporal con nombre que se define con `WITH` y se puede usar dentro de una consulta más grande.
Ventajas: legibilidad, reutilización, recursividad.

```sql
WITH nombre_cte AS (
    SELECT ...
)
SELECT * FROM nombre_cte;
```

In [0]:
%sql
-- Ejercicio 2.1: Usá una CTE para calcular el precio promedio en USD por zona,
-- y luego filtrá solo las zonas con más de 1000 propiedades.

WITH precio_por_zona AS (
    SELECT
        zona,
        AVG(CAST(precio AS DOUBLE)) AS precio_promedio_usd,
        COUNT(*) AS total_propiedades
    FROM bootcamp.bronze.properties_bronze
    WHERE moneda = 'USD' AND precio IS NOT NULL
    GROUP BY zona
)
SELECT *
FROM precio_por_zona
WHERE total_propiedades > 1000
ORDER BY precio_promedio_usd DESC;

In [0]:
%sql
-- Ejercicio 2.2: Usá múltiples CTEs para calcular el precio por metro cuadrado.
-- CTE 1: Filtrar propiedades en USD con metros válidos.
-- CTE 2: Calcular precio_m2 = precio / metros_cuadrados_cubiertos.
-- Query final: Top 10 zonas con mayor precio_m2 promedio.

WITH propiedades_validas AS (
    SELECT
        zona,
        CAST(precio AS DOUBLE) AS precio,
        CAST(metros_cuadrados_cubiertos AS DOUBLE) AS metros
    FROM bootcamp.bronze.properties_bronze
    WHERE moneda = 'USD'
      AND precio IS NOT NULL
      AND metros_cuadrados_cubiertos IS NOT NULL
      AND CAST(metros_cuadrados_cubiertos AS DOUBLE) > 0
),
precio_m2 AS (
    SELECT
        zona,
        precio / metros AS precio_por_m2,
        metros
    FROM propiedades_validas
)
SELECT
    zona,
    AVG(precio_por_m2) AS precio_m2_promedio,
    COUNT(*) AS total
FROM precio_m2
GROUP BY zona
HAVING COUNT(*) > 50
ORDER BY precio_m2_promedio DESC
LIMIT 10;

In [0]:
%sql
-- Ejercicio 2.3: Usá CTEs para comparar el precio promedio de alquiler vs venta por zona.
-- Mostrá zonas donde hay al menos 50 propiedades de cada tipo.

WITH alquileres AS (
    SELECT zona, AVG(CAST(precio AS DOUBLE)) AS precio_alquiler, COUNT(*) AS total
    FROM bootcamp.bronze.properties_bronze
    WHERE tipo_de_operacion = 'alquiler' AND precio IS NOT NULL
    GROUP BY zona
),
ventas AS (
    SELECT zona, AVG(CAST(precio AS DOUBLE)) AS precio_venta, COUNT(*) AS total
    FROM bootcamp.bronze.properties_bronze
    WHERE tipo_de_operacion = 'venta' AND precio IS NOT NULL
    GROUP BY zona
)
SELECT
    a.zona,
    a.precio_alquiler,
    v.precio_venta,
    v.precio_venta / a.precio_alquiler AS ratio_venta_alquiler,
    a.total AS total_alquileres,
    v.total AS total_ventas
FROM alquileres a
JOIN ventas v ON a.zona = v.zona
WHERE a.total >= 50 AND v.total >= 50
ORDER BY ratio_venta_alquiler DESC;

## PARTE 3: Window Functions

Las window functions realizan cálculos sobre un conjunto de filas relacionadas (una "ventana") sin colapsarlas en una sola fila.

```sql
funcion() OVER (
    PARTITION BY columna  -- agrupar
    ORDER BY columna      -- ordenar dentro del grupo
)
```

Funciones comunes: `ROW_NUMBER()`, `RANK()`, `DENSE_RANK()`, `LAG()`, `LEAD()`, `SUM() OVER`, `AVG() OVER`

In [0]:
%sql
-- Ejercicio 3.1: Usá ROW_NUMBER() para encontrar la propiedad más cara de cada zona (en USD).
-- Mostrá zona, precio, ubicacion y ambientes de esas propiedades.

WITH propiedades_usd AS (
    SELECT
        zona,
        ubicacion,
        CAST(precio AS DOUBLE) AS precio_num,
        ambientes,
        ROW_NUMBER() OVER (
            PARTITION BY zona
            ORDER BY CAST(precio AS DOUBLE) DESC
        ) AS rn
    FROM bootcamp.bronze.properties_bronze
    WHERE moneda = 'USD' AND precio IS NOT NULL
)
SELECT zona, ubicacion, precio_num AS precio, ambientes
FROM propiedades_usd
WHERE rn = 1
ORDER BY precio_num DESC
LIMIT 20;

In [0]:
%sql
-- Ejercicio 3.2: Para la zona 'palermo', rankeá las propiedades por precio usando RANK() y DENSE_RANK().
-- ¿Cuál es la diferencia entre RANK y DENSE_RANK cuando hay empates?

SELECT
    ubicacion,
    CAST(precio AS DOUBLE) AS precio,
    RANK() OVER (ORDER BY CAST(precio AS DOUBLE) DESC) AS ranking_rank,
    DENSE_RANK() OVER (ORDER BY CAST(precio AS DOUBLE) DESC) AS ranking_dense
FROM bootcamp.bronze.properties_bronze
WHERE zona = 'palermo'
  AND moneda = 'USD'
  AND precio IS NOT NULL
ORDER BY precio DESC
LIMIT 20;

In [0]:
%sql
-- Ejercicio 3.3: Usá LAG() y LEAD() para comparar el precio de cada propiedad con la anterior y la siguiente.
-- Filtrá por zona 'belgrano', moneda USD, y ordená por precio.
-- Calculá la diferencia de precio con la propiedad anterior.

SELECT
    ubicacion,
    CAST(precio AS DOUBLE) AS precio,
    LAG(CAST(precio AS DOUBLE)) OVER (ORDER BY CAST(precio AS DOUBLE)) AS precio_anterior,
    LEAD(CAST(precio AS DOUBLE)) OVER (ORDER BY CAST(precio AS DOUBLE)) AS precio_siguiente,
    CAST(precio AS DOUBLE) - LAG(CAST(precio AS DOUBLE)) OVER (ORDER BY CAST(precio AS DOUBLE)) AS diferencia_anterior
FROM bootcamp.bronze.properties_bronze
WHERE zona = 'belgrano'
  AND moneda = 'USD'
  AND precio IS NOT NULL
ORDER BY precio DESC
LIMIT 20;

In [0]:
%sql
-- Ejercicio 3.4: Calculá el promedio acumulado de precios por zona a medida que se recorren las propiedades.
-- Usá AVG() OVER con PARTITION BY zona y ORDER BY precio.

SELECT
    zona,
    ubicacion,
    CAST(precio AS DOUBLE) AS precio,
    AVG(CAST(precio AS DOUBLE)) OVER (
        PARTITION BY zona
        ORDER BY CAST(precio AS DOUBLE)
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS promedio_acumulado
FROM bootcamp.bronze.properties_bronze
WHERE moneda = 'USD'
  AND precio IS NOT NULL
  AND zona IN ('palermo', 'belgrano', 'recoleta')
ORDER BY zona, precio DESC
LIMIT 30;

In [0]:
%sql
-- Ejercicio 3.5: Calculá el porcentaje que representa cada propiedad respecto al total de precios de su zona.
-- Usá SUM() OVER (PARTITION BY zona) para obtener el total por zona.

SELECT
    zona,
    ubicacion,
    CAST(precio AS DOUBLE) AS precio,
    SUM(CAST(precio AS DOUBLE)) OVER (PARTITION BY zona) AS total_zona,
    ROUND(
        (CAST(precio AS DOUBLE) / SUM(CAST(precio AS DOUBLE)) OVER (PARTITION BY zona)) * 100,
        4
    ) AS porcentaje_del_total
FROM bootcamp.bronze.properties_bronze
WHERE moneda = 'USD'
  AND precio IS NOT NULL
  AND zona IN ('palermo', 'caballito', 'flores')
ORDER BY zona, porcentaje_del_total DESC
LIMIT 30;